# PneumoniaMNIST + MedSymmFlow — Synthetic Augmentation

**Does augmenting PneumoniaMNIST with MedSymmFlow-generated images improve a ResNet-18 classifier?**

Just run this top to bottom (**Runtime -> Run all**). Everything is automatic.

> **Runtime must be GPU:** Runtime -> Change runtime type -> **T4 GPU**.

The experiment logic lives in `project/augmentation.py` **in the repo**, so fixes arrive
via `git pull` — this notebook is only config, narrative, and calls.

| Section | What happens |
|---|---|
| 0 | Mount Drive, clone the repo, install deps, **record reproducibility fixtures** |
| 1 | Load the genuine `medmnist` splits (4708/524/624) and **self-test** the code |
| 2 | Baselines **B0/B1/B2** (real data only) |
| 3–4 | Generate synthetic X-rays, then filter them |
| 5 | Synthetic arms **S1/S2/S3** + **D1** (synthetic-only) |
| 6–7 | **C1** reference, results, statistics, distillation fingerprint |

**First run:** leave `quick=True` (a few minutes) to confirm everything works.
**Real run:** set `quick=False` in section 0 and re-run.

## 0. Setup

Nothing to edit here except the `quick` flag at the bottom of this section.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone (or update) the repo and install the packages Colab is missing.
import os, subprocess, sys

REPO = "/content/MedSymmFlow"
OUT  = "/content/drive/MyDrive/MedSymmFlow_Project"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone",
                    "https://github.com/RonNekrashevich/MedSymmFlow.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "main"], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)

DEPS = ["medmnist", "torchdiffeq", "diffusers", "accelerate", "zuko",
        "scikit-learn", "scipy", "loguru", "python-dotenv", "datasets"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], check=True)
print("repo + deps ready")

In [ ]:
# Reproducibility fixtures.
#
# train_classifier seeds the RNG and then builds the model, and Conv2d/Linear consume
# that RNG at construction -- so a refactor that reorders module creation would shift
# every result by an amount that looks exactly like a real effect. These fixtures are
# recorded from the PRE-refactor commit (a91816b) in a separate process, so section 1
# can prove the current code is numerically identical.
#
# Runs once; skipped forever after.
import json, os, subprocess
from pathlib import Path

FIX = Path(OUT) / "fixtures_prerefactor.json"
Path(OUT).mkdir(parents=True, exist_ok=True)

if FIX.exists():
    print("fixtures already recorded:", FIX)
    print(json.dumps(json.loads(FIX.read_text()), indent=2)[:400])
else:
    print("recording fixtures from the pre-refactor commit ...")
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "a91816b"], check=True)
    r = subprocess.run(["python", "project/record_fixtures.py", "--out", OUT],
                       cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-2500:] or r.stderr[-2500:])
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "main"], check=True)
    if not FIX.exists():
        print("\nWARNING: fixtures were not written. The self-test in section 1 will be\n"
              "skipped, but the experiment still runs normally.")

In [ ]:
# Preflight: import the MedSymmFlow chain exactly as the generator subprocess does,
# so a missing dependency fails here in seconds instead of after training.
import os, subprocess
probe = ("from medsymmflow.models.SymmFMClass import SymmFMClass; "
         "from medsymmflow.data.Dataloaders import pick_dataset; print('MedSymmFlow imports OK')")
res = subprocess.run(["python", "-c", probe], cwd=REPO,
                     env=dict(os.environ, PYTHONPATH=f"{REPO}/src:{REPO}/src/medsymmflow"),
                     capture_output=True, text=True)
print(res.stdout.strip() or res.stderr[-1500:])
assert res.returncode == 0, "MedSymmFlow import chain is broken"

In [ ]:
import sys
sys.path.insert(0, f"{REPO}/project")
from augmentation import Experiment, Config

# ---------------------------------------------------------------------------
#  quick=True   fast smoke test  (1 seed, 5 epochs, budget 500)  <- start here
#  quick=False  the real run     (5 seeds, 15 epochs, full sweep)
# ---------------------------------------------------------------------------
QUICK = True

cfg = Config(
    quick=QUICK,
    save_dir=OUT,
    medsymm_root=REPO,
    **({} if QUICK else dict(budgets=[250, 500, 1000, 2000, 4708], seeds=[0, 1, 2, 3, 4])),
)
exp = Experiment(cfg)

## 1. Data and self-test

Loads the **genuine** `medmnist` splits (the MedSymmFlow repo's own loaders alias
validation to test — we sidestep them), asserts 4708/524/624, and reports the
train/test prevalence shift, which is why AUC rather than accuracy is the primary metric.

In [ ]:
prevalence = exp.setup_data()
display(prevalence)

In [ ]:
# Must print "numerically inert". If it says FAIL, stop and report it -- the results
# would not be comparable to the earlier run.
exp.selftest_repro(strict=False)

## 2. Baselines — B0 / B1 / B2

All real-data baselines, so a synthetic arm has to beat the *strongest* of them:
**B0** plain, **B1** class-weighted loss, **B2** minority oversampling.
B0 must land near the published **0.944** reproduction target.

In [ ]:
exp.run_baselines()

## 3. Synthetic generation — MedSymmFlow @ 28 px

Downloads the pretrained weights (Zenodo, ~755 MB, cached) and samples class-conditioned
images. The RGB_28 checkpoint is sampled at 32 px — its true training resolution, and the
UNet needs a size divisible by 8 — then stored at 28 px to match the real data.
Everything persists to Drive, so this is skipped on later runs.

In [ ]:
exp.download_weights()
exp.generate_synthetic()
exp.visualize_samples()   # should look like chest X-rays

## 4. Filtering

Two screens before any synthetic image is used:
**memorisation** (drop near-copies of real training images) and **confidence**
(keep only samples a real-data model scores confidently).

The confidence scorer is now the arm's **own** model, so no information from a larger
budget leaks into a small-budget arm — and the filtered set is cached under a
configuration hash, so it is identical no matter which budgets are in the sweep.

In [ ]:
filter_summary = exp.filter_synthetic()

## 5. Synthetic arms — S1 / S2 / S3, and D1

**S1** pretrain on synthetic then fine-tune on real · **S2** mix real + synthetic ·
**S3** synthetic only to rebalance classes · **D1** train on synthetic *only* (the
diagnostic: if it scores well, the generator captured the disease itself).

In [ ]:
exp.run_synthetic()

In [ ]:
exp.run_diagnostic_d1()

## 6. C1 — the MedSymmFlow reference

MedSymmFlow classifies as well as it generates, so it is the control: if a synthetic arm
wins, we must check the gain is not just inherited from the generator. We now **measure**
it on our own test split rather than quoting the paper's number.

In [ ]:
exp.measure_c1()
exp.record_c1()

## 7. Results

Mean test AUC with 95% confidence intervals across seeds. A synthetic arm counts as
effective only if it beats the strongest baseline with non-overlapping CIs **and** the
gain is not explained by C1.

In [ ]:
summary, comparison = exp.summarize()
display(summary)
print("\nSynthetic arms vs strongest baseline:")
display(comparison)
exp.plot(summary)

In [ ]:
# Paired seed-wise test (removes shared subsample/init variance) + Benjamini-Hochberg
# correction. Needs >= 2 seeds, so it only reports on the real run.
paired = exp.paired_tests()
display(paired)

## 7b. Distillation fingerprint

Does a synthetic-trained model copy MedSymmFlow's **mistakes** more than a real-trained
one? Copying specific errors requires copying the decision function, which mere
data-manifold coverage cannot produce. Read the `*_matched` columns — they remove the
class-prior confound between the two models.

In [ ]:
fingerprint = exp.distillation_agreement()
display(fingerprint)

## 8. Next steps

- **Real run:** set `QUICK = False` in section 0 and re-run. It is resumable — a dead
  session costs one cell, not the sweep.
- **Filter ablation (nearly free, reuses the caches):** is keeping *confident* samples
  actually better than keeping the *hard* ones?

  ```python
  for mode in ["none", "keep_uncertain", "random_match"]:
      exp.cfg.filter_mode = mode
      exp.cfg.run_tag = f"filter={mode}"
      exp._filter_cache.clear()
      exp.filter_synthetic(plot=False)
      exp.run_synthetic()
  ```
- **Generation sweep:** `exp.cfg.gen_beta` in {1, 2, 4, 6} — tests whether sharper class
  conditioning helps downstream (the free version of "add a classification loss").

**Editing the logic:** change `project/augmentation.py` in the repo, re-run section 0,
then **Runtime -> Restart session** and re-run. No notebook re-upload needed.